# Антибот: оценка вероятности принадлежности к боту по событиям суточного окна

**Задача.** По событиям `events.csv.gz` внутри суточного окна наблюдения выдать для каждой
`cookie_id` из `test.csv` `score` от 0 до 1: чем выше, тем вероятнее, что куки — трафик
сервисов автоматического сбора данных.

**Метрика.** Precision при Recall ≥ 0.70 (официальный код — `metric.py`). Важные детали
механики: куки сортируются по score, одинаковые score берутся **одной группой**
(нельзя разрезать), результат — максимальный precision среди порогов с recall ≥ 0.70.

**Ограничения, которые соблюдаем:** только события внутри окна
(`window_start_ts <= event_ts < window_end_ts`), никаких внешних API и LLM >1B,
никакой ручной разметки теста и хардкода по `cookie_id`, `random_state = 42`.

**Идея решения.** На события окна наливаем четыре слоя числовых признаков
(интенсивность и ритм потока, состав поведения, разнообразие/глубина, среда и метаданные
плюс перцентили внутри того же дня), добавляем слой **категориальных** признаков и
ранжируем куки ансамблем из 7 моделей: 4 числовых LightGBM + 2 LightGBM и 1 CatBoost
на «числах + категориях». Ансамбль берёт взвешенное среднее **процентилей** скоров,
поэтому модели с разной калибровкой сравнимы между собой, а разные «взгляды» на
поведение складываются.

**Как читать ноутбук** (пишу последовательно, каждый блок проверяет предыдущий):

1. Загрузка данных и осмотр — что вообще лежит в событиях;
2. Окно наблюдения — отбираем события строго внутри окна;
3. Признаки — пять слоёв, каждый печатает форму и пример результата;
4. Диагностика признаков — одномерный AUC, что реально разделяет классы;
5. Валидация — walk-forward по 7 дням, официальная метрика, сравнение моделей
   и блендингов;
6. Финальное обучение на всём train и запись `submission.csv` с assert-проверками;
7. Что показали эксперименты — что взял, что отбросил и почему.

In [2]:
import os
import sys
import time
import warnings

import numpy as np
import pandas as pd

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

T0 = time.time()


def elapsed() -> str:
    return f'{time.time() - T0:.0f}s'


DATA = 'data'
OUT = 'submission.csv'

print('python', sys.version.split()[0])
print('pandas', pd.__version__, '| numpy', np.__version__)
try:
    import lightgbm as lgb
    print('lightgbm', lgb.__version__)
except Exception as e:
    print('lightgbm недоступен:', e)

python 3.13.14
pandas 2.3.3 | numpy 2.3.5
lightgbm 4.7.0


In [3]:
from metric import precision_at_recall   # официальная реализация метрики

train = pd.read_csv(f'{DATA}/train.csv',
                    parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv(f'{DATA}/test.csv',
                   parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv(f'{DATA}/events.csv.gz', parse_dates=['event_ts'])

print('train', train.shape, '| test', test.shape, '| events', events.shape)
print('доля ботов в train:', round(train.target.mean(), 4))
print('окна train:', train.window_start_ts.min().date(), '->', train.window_start_ts.max().date())
print('окна test :', test.window_start_ts.min().date(), '->', test.window_start_ts.max().date())
train.head()

train (11091, 5) | test (4909, 4) | events (328905, 14)
доля ботов в train: 0.0811
окна train: 2026-04-06 -> 2026-04-19
окна test : 2026-04-20 -> 2026-04-26


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


## 1. Осмотр данных

Прежде чем строить признаки, смотрим, с чем имеем дело. Что важно узнать заранее:

* какие типы событий бывают и как они распределены — из этого растут признаки «состав потока»;
* `platform` приходит в **четырёх регистрах** (`web/WEB/Web/desktop`) — если не нормализовать,
  один и тот же веб разбивается на четыре «платформы»;
* `user_agent` включает в том числе `HeadlessChrome` — прямой признак автоматизации;
* есть ли пропуски, дубликаты, и не отсортирован ли файл событий по времени.

Начнём с состава событий.

In [5]:
print('типы событий и их частота:')
print(events.event_name.value_counts(), '\n')

print('доля событий с пропусками по колонкам:')
print(events.isna().mean().round(3))

типы событий и их частота:
event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

доля событий с пропусками по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


In [6]:
# платформа: те же значения в разных регистрах — нормализуем при сборке признаков
print('значения platform (сырые):')
print(events.platform.value_counts(), '\n')
print('после приведения к нижнему регистру:')
print(events.platform.str.lower().replace({'iphone': 'ios'}).value_counts())

значения platform (сырые):
platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

после приведения к нижнему регистру:
platform
web        138792
android    126875
desktop     46866
ios         16372
Name: count, dtype: int64


In [7]:
print('глобально отсортировано по времени:', events.event_ts.is_monotonic_increasing)
print('внутри куки отсортировано:',
      events.sort_values(['cookie_id', 'event_ts']).groupby('cookie_id').event_ts
      .apply(lambda s: s.is_monotonic_increasing).all())
print('точных дублей строк:', events.duplicated().sum())

глобально отсортировано по времени: False
внутри куки отсортировано: True
точных дублей строк: 4863


In [8]:
print('куки без единого события: train',
      (~train.cookie_id.isin(events.cookie_id)).sum(),
      '| test', (~test.cookie_id.isin(events.cookie_id)).sum())
print('строк в метаданных: train', len(train), '| test', len(test))

куки без единого события: train 0 | test 0
строк в метаданных: train 11091 | test 4909


## 2. Окно наблюдения

Главное ограничение задачи: признаки считаются **только по событиям внутри окна**
`window_start_ts <= event_ts < window_end_ts`. Это значит:

* всё, что произошло до открытия окна или после его закрытия, модель не видит
  (включая показы капчи — внутри окна их не бывает вообще, поэтому «captcha_shown»
  в признаки не попадает);
* доступная информация — только то, что известно к `window_end_ts`;
* для теста даты и состав кук заранее неизвестны, поэтому ни один признак
  не завязан на конкретный день или идентификатор.

Отфильтруем события и посмотрим, сколько их остаётся и как объём событий
связан с целевой переменной (первый «сырой» сигнал: боты активнее).

In [10]:
def events_in_window(events: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """События, попавшие в суточное окно наблюдения куки, отсортированные внутри куки."""
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    ev = ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]
    return ev.sort_values(['cookie_id', 'event_ts']).reset_index(drop=True)


ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print('событий внутри окна: train', len(ev_tr), '| test', len(ev_te))
print('всего событий было:', len(events), '→ потеряли внеоконных:',
      len(events) - len(ev_tr) - len(ev_te))
print('куки с событиями: train', ev_tr.cookie_id.nunique(),
      '| test', ev_te.cookie_id.nunique())

событий внутри окна: train 198436 | test 89690
всего событий было: 328905 → потеряли внеоконных: 40779
куки с событиями: train 11091 | test 4909


In [11]:
# первый сигнал: чем больше событий в окне, тем вероятнее бот
tmp = (train[['cookie_id', 'target']]
       .merge(ev_tr.groupby('cookie_id').size().rename('n'), on='cookie_id', how='left')
       .fillna({'n': 0}))
print('доля ботов по числу событий в окне:')
print(tmp.groupby(pd.cut(tmp.n, [0, 5, 15, 40, 1000]))['target'].agg(['size', 'mean']).round(3))

доля ботов по числу событий в окне:
            size   mean
n                      
(0, 5]      2311  0.035
(5, 15]     4333  0.062
(15, 40]    3390  0.101
(40, 1000]  1057  0.197


## 3. Признаки

Четыре слоя **числовых** признаков (плюс категориальный слой — секция 3.6), все
считаются по событиям окна и метаданным куки, доступным к `window_end_ts`:

1. **Ядро (per-cookie).** Объёмы и доли каждого типа событий, разнообразие
   (объявления/категории/локации/запросы/страницы), статистика пауз (сессии, квантили,
   регулярность), ритм по часам, положение активности внутри окна, курсор, разбор
   `user_agent`, переходы между событиями, дубликаты.
2. **Скриптовое поведение.** Монотонность пагинации, кратность пауз «круглым» числам,
   повторы запросов, возвраты к товарам, отсутствие «живых» действий (логин, избранное,
   сообщения), размах активности по часам.
3. **Доменные без меток.** Популярность объекта/запроса/категории/локации среди кук того же
   дня, смена регистра платформы, прыжки между локациями, позиция курсора по типам событий.
4. **Среда и метаданные.** Микс платформ и типов UA, доля `pro`-продавцов, топовые доли
   категории/локации/запроса, возраст куки, день недели окна.

Плюс **перцентили и z-оценки внутри того же суточного дня** — метки не используются,
поэтому признак честно доступен в момент окончания окна.

Начнём с констант и общих хелперов.

In [13]:
EVENT_TYPES = ['item_view', 'search_results_view', 'photo_swipe', 'favorite_add',
               'seller_page_view', 'contact_phone_show', 'contact_chat_open',
               'contact_message_sent', 'login', 'captcha_shown']
CONTACT = {'contact_phone_show', 'contact_chat_open', 'contact_message_sent'}
SESSION_GAP = 300          # граница сессии, сек
DAY_BINS = [(0, 6), (6, 10), (10, 14), (14, 18), (18, 24)]
PLAT_LEVELS = ['android', 'desktop', 'ios', 'web']
UA_KINDS = ['headless', 'yabrowser', 'firefox', 'chrome', 'safari_ios', 'other']

print('типов событий:', len(EVENT_TYPES))
print('бины по часам:', DAY_BINS)

типов событий: 10
бины по часам: [(0, 6), (6, 10), (10, 14), (14, 18), (18, 24)]


In [14]:
def ent(counts) -> float:
    """Энтропия по счётчикам: насколько распределение равномерно.

    Используем везде, где считаем «разнообразие»: типы событий, локации, паузы.
    Равномерное распределение → большая энтропия, одно доминирующее значение → ≈0.
    """
    p = np.asarray(counts, dtype=float)
    p = p[p > 0]
    if not len(p):
        return 0.0
    p = p / p.sum()
    return float(-(p * np.log(p)).sum())


# самопроверка хелпера: энтропия двух равных значений = log(2)
assert abs(ent([5, 5]) - np.log(2)) < 1e-9 and ent([10, 0]) == 0.0
print('ent([5, 5]) =', round(ent([5, 5]), 4), '| ent([10, 1]) =', round(ent([10, 1]), 4))

ent([5, 5]) = 0.6931 | ent([10, 1]) = 0.3046


In [15]:
def ua_kind(ua: str) -> str:
    """Семейство браузера одной строкой — удобно и для долей, и как категория."""
    if 'Headless' in ua:
        return 'headless'
    if 'YaBrowser' in ua:
        return 'yabrowser'
    if 'Firefox' in ua:
        return 'firefox'
    if 'Chrome' in ua:
        return 'chrome'
    if 'iPhone' in ua or 'iPad' in ua:
        return 'safari_ios'
    return 'other'


for sample in ['Mozilla/5.0 HeadlessChrome/120', 'Mozilla/5.0 Chrome/121 Windows',
               'Mozilla/5.0 iPhone Safari']:
    print(f'{sample[:40]:42s} → {ua_kind(sample)}')

Mozilla/5.0 HeadlessChrome/120             → headless
Mozilla/5.0 Chrome/121 Windows             → chrome
Mozilla/5.0 iPhone Safari                  → safari_ios


Теперь сам «движок» ядра. Он устроен в два шага, чтобы не тормозить на 16k кук:

1. **`prepare_arrays`** — один раз кодируем тяжёлые колонки событий в numpy
   (`factorize` строк в числа, время в секунды); на выходе — словарь массивов
   и индекс «кука → номера строк»;
2. **`cookie_features`** — чистый python-цикл по кукам, внутри только numpy:
   на каждую куку собираем словарь признаков.

Такой разбиение — не «функции ради функций»: шаг 1 выполняется один раз на весь
датафрейм, шаг 2 вызывается 16 000 раз, и именно поэтому он должен работать на
массивах, а не на pandas-группировках.

In [17]:
def prepare_arrays(w: pd.DataFrame) -> dict:
    """Один раз кодируем колонки событий, чтобы цикл по кукам шёл на numpy."""
    a = {}
    a['ts'] = w.event_ts.values.astype('datetime64[s]').astype(np.int64)
    a['hour'] = w.event_ts.dt.hour.values
    # кодируем события по фиксированному EVENT_TYPES, а не по порядку появления строк:
    # иначе train и test дают разные номера одному и тому же event_name, а ниже мы
    # читаем счётчики через EVENT_TYPES.index(...) — и признаки становятся про другое
    a['eid'] = pd.Categorical(w.event_name, categories=EVENT_TYPES).codes.astype(np.int64)
    a['item'] = w.item_id.values.astype(float)
    a['cat'], _ = pd.factorize(w.item_category)
    a['loc'], _ = pd.factorize(w.item_location)
    a['sell'], _ = pd.factorize(w.seller_type)
    a['query'], _ = pd.factorize(w.search_query)
    a['page'] = w.search_page.values.astype(float)
    a['plat'], _ = pd.factorize(w.platform)
    a['plat_l'], _ = pd.factorize(w.platform.str.lower())
    a['ua'], _ = pd.factorize(w.user_agent)
    a['px'] = w.pointer_x.values.astype(float)
    a['py'] = w.pointer_y.values.astype(float)
    a['qstr'] = w.search_query.fillna('').values
    a['qlen'] = w.search_query.str.len().values.astype(float)
    a['idx'] = w.groupby('cookie_id', sort=False).indices
    g = w.groupby('cookie_id', sort=False)
    a['ws_d'] = {k: v.value // 10 ** 9 for k, v in g.window_start_ts.first().items()}
    a['we_d'] = {k: v.value // 10 ** 9 for k, v in g.window_end_ts.first().items()}
    a['ua_d'] = g.user_agent.first().to_dict()
    return a


def cookie_features(cid: str, a: dict) -> dict:
    """Все признаки одной куки по её событиям внутри окна."""
    idx = a['idx'].get(cid)
    if idx is None:
        return {'cookie_id': cid, 'n_events': 0}
    ts = a['ts'][idx]
    n = len(idx)
    f = {'n_events': n}

    eid = a['eid'][idx]
    item = a['item'][idx]
    cat = a['cat'][idx]
    loc = a['loc'][idx]
    sell = a['sell'][idx]
    query = a['query'][idx]
    page = a['page'][idx]
    hour = a['hour'][idx]
    px, py = a['px'][idx], a['py'][idx]

    #состав потока событий
    cnt = np.bincount(eid, minlength=len(EVENT_TYPES))
    for k, name in enumerate(EVENT_TYPES):
        f[f'n_{name}'] = int(cnt[k])
        f[f'r_{name}'] = float(cnt[k]) / n
    f['n_evt_types'] = int((cnt > 0).sum())
    f['evt_entropy'] = ent(cnt)

    #разнообразие
    def entropy_of(v):
        v = pd.Series(v)
        v = v[v >= 0]                       # -1 == NaN после factorize
        if not len(v):
            return 0.0
        _, c = np.unique(v, return_counts=True)
        return ent(c)

    def n_uniq(v):
        """Сколько разных значений; −1 (NaN после factorize) не считаем, как в entropy_of."""
        v = np.asarray(v)
        v = v[v >= 0]
        return int(np.unique(v).size)

    f['n_items'] = int(pd.Series(item).nunique())   # item — float с NaN, nunique их сам отбрасывает
    f['n_cats'] = n_uniq(cat)
    f['n_locs'] = n_uniq(loc)
    f['n_sellers'] = n_uniq(sell)
    f['n_queries'] = n_uniq(query)
    f['n_plat_case'] = n_uniq(a['plat'][idx])
    f['n_plat'] = n_uniq(a['plat_l'][idx])
    f['n_ua'] = n_uniq(a['ua'][idx])
    f['cat_entropy'] = entropy_of(cat)
    f['loc_entropy'] = entropy_of(loc)
    f['query_entropy'] = entropy_of(query)
    f['plat_entropy'] = entropy_of(a['plat'][idx])

    #дубликаты
    mat = np.stack([eid, ts, np.nan_to_num(item, nan=-1), cat, loc, sell, query,
                    np.nan_to_num(page, nan=-1), np.nan_to_num(px, nan=-1),
                    np.nan_to_num(py, nan=-1)], axis=1)
    n_uniq = int(np.unique(mat, axis=0).shape[0])
    f['n_uniq_rows'] = n_uniq
    f['dup_row_rate'] = 1.0 - n_uniq / n
    f['dup_ts_rate'] = 1.0 - len(np.unique(ts)) / n

    #повторы и конверсии
    views = int(cnt[EVENT_TYPES.index('item_view')])
    # item_repeat — доля повторных просмотров: уникальные товары считаем только среди
    # самих item_view, иначе n_items (уникальные товары по ВСЕМ событиям с item_id)
    # оказывается больше числа просмотров и признак уходил в минус
    view_items = item[eid == EVENT_TYPES.index('item_view')]
    view_items = view_items[~np.isnan(view_items)]
    f['item_repeat'] = (1.0 - np.unique(view_items).size / len(view_items)
                        if len(view_items) else 0.0)
    f['items_per_evt'] = f['n_items'] / n
    f['cats_per_evt'] = f['n_cats'] / n
    f['locs_per_evt'] = f['n_locs'] / n
    f['queries_per_evt'] = f['n_queries'] / n
    # просмотров может не быть вовсе — тогда отношения к ним не определены, берём 0
    if views:
        f['swipe_per_view'] = float(cnt[EVENT_TYPES.index('photo_swipe')]) / views
        f['fav_per_view'] = float(cnt[EVENT_TYPES.index('favorite_add')]) / views
        f['seller_per_view'] = float(cnt[EVENT_TYPES.index('seller_page_view')]) / views
    else:
        f['swipe_per_view'] = f['fav_per_view'] = f['seller_per_view'] = 0.0
    f['contact_n'] = int(sum(cnt[EVENT_TYPES.index(e)] for e in CONTACT))
    f['r_contact'] = f['contact_n'] / n
    f['has_login'] = int(cnt[EVENT_TYPES.index('login')] > 0)

    #поиск
    ok_page = ~np.isnan(page)
    if ok_page.any():
        p = page[ok_page]
        f['n_search_pages'] = int(len(np.unique(p)))
        f['page_max'] = float(p.max())
        f['page_mean'] = float(p.mean())
        f['page_std'] = float(p.std()) if len(p) > 1 else 0.0
        f['page_deep'] = float((p >= 5).mean())
        f['page_step_mean'] = float(np.abs(np.diff(np.sort(p))).mean()) if len(p) > 1 else 0.0
    else:
        for k in ['n_search_pages', 'page_max', 'page_mean', 'page_std', 'page_deep',
                  'page_step_mean']:
            f[k] = 0.0
    qn = query[query >= 0]
    f['query_repeat'] = 1.0 - len(np.unique(qn)) / len(qn) if len(qn) else 0.0
    ql = a['qlen'][idx]
    ok_q = ~np.isnan(ql)
    f['query_len_mean'] = float(ql[ok_q].mean()) if ok_q.any() else 0.0
    f['query_len_std'] = float(ql[ok_q].std()) if ok_q.sum() > 1 else 0.0
    # доля запросов кириллицей (пишут по-русски — в основном люди)
    qstr = a['qstr'][idx][ok_q]
    f['query_cyr'] = (float(np.mean([any('а' <= ch <= 'я' or ch == 'ё' for ch in s)
                                     for s in qstr])) if ok_q.any() else 0.0)

    #тайминг
    gaps = np.diff(ts)
    f['span_s'] = int(ts[-1] - ts[0])
    if len(gaps):
        f['gap_mean'] = float(gaps.mean())
        f['gap_std'] = float(gaps.std())
        f['gap_med'] = float(np.median(gaps))
        f['gap_min'] = float(gaps.min())
        f['gap_max'] = float(gaps.max())
        f['gap_cv'] = float(gaps.std() / (gaps.mean() + 1e-9))
        f['gap_burst'] = float((gaps.std() - gaps.mean()) / (gaps.std() + gaps.mean() + 1e-9))
        for q in (0.25, 0.75, 0.9):
            f[f'gap_q{int(q * 100)}'] = float(np.quantile(gaps, q))
        for th in (1, 3, 10, 60):
            f[f'gap_lt{th}'] = float((gaps < th).mean())
        f['gap_gt300'] = float((gaps > SESSION_GAP).mean())
        f['gap_gt3600'] = float((gaps > 3600).mean())
        gv, gc = np.unique(gaps, return_counts=True)
        f['gap_nuniq'] = int(len(gv))
        f['gap_top_share'] = float(gc.max() / len(gaps))
        f['gap_entropy'] = ent(gc)
        f['gap_eq_med'] = float((gaps == np.median(gaps)).mean())
    else:
        for k in ['gap_mean', 'gap_std', 'gap_med', 'gap_min', 'gap_max', 'gap_cv', 'gap_burst',
                  'gap_q25', 'gap_q75', 'gap_q90', 'gap_lt1', 'gap_lt3', 'gap_lt10', 'gap_lt60',
                  'gap_gt300', 'gap_gt3600', 'gap_nuniq', 'gap_top_share', 'gap_entropy',
                  'gap_eq_med']:
            f[k] = 0.0

    #сессии
    is_new = np.r_[True, gaps > SESSION_GAP]
    sess = np.cumsum(is_new)
    sizes = np.bincount(sess)[1:]
    f['n_sessions'] = int(sess[-1])
    f['sess_max'] = int(sizes.max())
    f['sess_mean'] = float(sizes.mean())
    f['sess_share_max'] = float(sizes.max() / n)
    first = np.searchsorted(sess, np.unique(sess), side='left')
    last = np.searchsorted(sess, np.unique(sess), side='right') - 1
    dur = ts[last] - ts[first]
    f['sess_dur_mean'] = float(dur.mean())
    f['sess_dur_max'] = float(dur.max())
    f['sess_per_hour'] = f['n_sessions'] / max(len(np.unique(hour)), 1)

    #ритм по часам
    hb = np.array([((hour >= lo) & (hour < hi)).sum() for lo, hi in DAY_BINS])
    for (lo, hi), c in zip(DAY_BINS, hb):
        f[f'hr_{lo}_{hi}'] = float(c) / n
    f['hr_entropy'] = ent(hb)
    f['hour_mean'] = float(hour.mean())
    f['hour_std'] = float(hour.std()) if n > 1 else 0.0
    f['night_share'] = float(((hour >= 0) & (hour < 6)).mean())
    f['work_share'] = float(((hour >= 9) & (hour < 19)).mean())
    f['n_hours'] = int(len(np.unique(hour)))
    f['n_minutes'] = int(pd.Series(ts // 60).nunique())
    f['events_per_active_hour'] = n / max(f['n_hours'], 1)
    f['events_per_minute'] = n / max(f['n_minutes'], 1)

    #положение активности внутри окна
    ws, we = a['ws_d'][cid], a['we_d'][cid]
    day_len = max(we - ws, 1)
    f['t_first'] = float(ts[0] - ws) / day_len
    f['t_last'] = float(we - ts[-1]) / day_len
    f['span_frac'] = f['span_s'] / day_len

    #курсор
    has_p = ~np.isnan(px)
    f['pointer_rate'] = float(has_p.mean())
    if has_p.any():
        X, Y = px[has_p], py[has_p]
        f['px_mean'], f['py_mean'] = float(X.mean()), float(Y.mean())
        f['px_std'] = float(X.std()) if len(X) > 1 else 0.0
        f['py_std'] = float(Y.std()) if len(Y) > 1 else 0.0
        f['px_range'] = float(X.max() - X.min())
        f['py_range'] = float(Y.max() - Y.min())
        f['pointer_nuniq'] = len(np.unique(np.stack([X, Y], 1), axis=0)) / len(X)
        f['pointer_round'] = float(((X % 10 == 0) & (Y % 10 == 0)).mean())
    else:
        for k in ['px_mean', 'py_mean', 'px_std', 'py_std', 'px_range', 'py_range',
                  'pointer_nuniq', 'pointer_round']:
            f[k] = 0.0

    #user agent: флаги по подстрокам (инлайн, чтобы не плодить одноразовые хелперы)
    ua = a['ua_d'][cid]
    f['ua_headless'] = int('Headless' in ua)
    f['ua_yabrowser'] = int('YaBrowser' in ua)
    f['ua_firefox'] = int('Firefox' in ua)
    f['ua_chrome'] = int('Chrome' in ua and 'Headless' not in ua)
    f['ua_windows'] = int('Windows' in ua)
    f['ua_mac'] = int('Macintosh' in ua)
    f['ua_linux'] = int('X11' in ua)
    f['ua_android'] = int('Android' in ua)
    f['ua_iphone'] = int(('iPhone' in ua) or ('iPad' in ua))
    f['ua_desktop'] = int(('Windows' in ua) or ('Macintosh' in ua) or ('X11' in ua))

    #переходы между событиями
    if n > 1:
        a1, a2 = eid[:-1], eid[1:]
        f['self_trans'] = float((a1 == a2).mean())
        big = a1.astype(np.int64) * len(EVENT_TYPES) + a2
        bv, bc = np.unique(big, return_counts=True)
        f['n_bigrams'] = int(len(bv))
        f['bigram_entropy'] = ent(bc)
        f['same_item_trans'] = float(((item[:-1] == item[1:]) & ~np.isnan(item[:-1])).mean())
        is_search = a1 == EVENT_TYPES.index('search_results_view')
        f['search_to_item'] = (float((a2[is_search] == EVENT_TYPES.index('item_view')).mean())
                               if is_search.any() else 0.0)
        is_view = a1 == EVENT_TYPES.index('item_view')
        f['view_to_swipe'] = (float((a2[is_view] == EVENT_TYPES.index('photo_swipe')).mean())
                              if is_view.any() else 0.0)
        f['seq_regularity'] = float(pd.Series(big).value_counts(normalize=True).iloc[0])
    else:
        for k in ['self_trans', 'n_bigrams', 'bigram_entropy', 'same_item_trans',
                  'search_to_item', 'view_to_swipe', 'seq_regularity']:
            f[k] = 0.0
    return f

Собираем всё в матрицу: применяем `cookie_features` к каждой куке из метаданных
(даже если у куки нет ни одного события — для неё вернётся нулевая строка),
сохраняем порядок метаданных и добавляем «моментальные» метки куки: возраст,
час/день недели создания, день недели окна. Всё это известно к `window_end_ts`.

In [19]:
def core_features(w: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    a = prepare_arrays(w)
    rows = [{**cookie_features(cid, a), 'cookie_id': cid} for cid in meta.cookie_id]
    X = pd.DataFrame(rows)
    X = meta[['cookie_id']].merge(X, on='cookie_id', how='left')
    assert X.cookie_id.equals(meta.cookie_id), 'порядок кук изменился'

    # метаданные куки (известны к окончанию окна)
    X['cookie_age_days'] = (meta.window_start_ts - meta.cookie_created_at).dt.total_seconds() / 86400
    X['cookie_created_hour'] = meta.cookie_created_at.dt.hour.values
    X['cookie_created_dow'] = meta.cookie_created_at.dt.dayofweek.values
    X['window_dow'] = meta.window_start_ts.dt.dayofweek.values
    return X.fillna(0.0).replace([np.inf, -np.inf], 0.0)


print('ядро признаков определено', elapsed())

ядро признаков определено 5s


### 3.1 Скриптовое поведение

Люди и боты листают выдачу по-разному. Что ловим здесь:

* **пагинация** — человек листает вперёд монотонно и почти всегда на шаг +1, бот часто
  прыгает, перескакивает, возвращается назад или вообще не использует `search_page`;
* **«круглые» паузы** — автоскрипт спит фиксированные 5/10/60 секунд, у человека
  интервалы произвольные и редко кратны круглым числам;
* **повторы** — бот крутит одни и те же запросы и возвращается к тем же товарам;
* **отсутствие живых действий** — логин, избранное, сообщения бот обычно не делает:
  он «потребляет» выдачу, но не участвует в жизни сервиса;
* **размах активности** — активность бота распределена по часам ровнее
  (нет человеческих провалов ночью и пиков вечером).

In [21]:
def script_features(w: pd.DataFrame) -> pd.DataFrame:
    pid = w.groupby('cookie_id', sort=False).indices
    page, item, q = w.search_page.values, w.item_id.values, w.search_query.values
    ts = w.event_ts.values.astype('datetime64[s]').astype(np.int64)
    nm = w.event_name.values
    out = {}

    for cid, idx in pid.items():
        n = len(idx)
        f = {}

        p = page[idx]
        pk = ~np.isnan(p)
        pp = p[pk]
        if len(pp) > 1:
            d = np.diff(pp)
            f['page_mono'] = float((d > 0).mean())
            f['page_step1'] = float((d == 1).mean())
            f['page_back'] = float((d < 0).mean())
            f['page_span_ratio'] = float((pp.max() - pp.min()) / max(len(pp) - 1, 1))
        else:
            f['page_mono'] = f['page_step1'] = f['page_back'] = f['page_span_ratio'] = 0.0

        qq = q[idx]
        both = (~pd.isna(qq)) & pk
        f['uniq_qp'] = (float(len(set(zip(qq[both], p[both]))) / both.sum())
                        if both.sum() > 1 else 0.0)
        f['n_search_evt'] = int(both.sum())

        t = ts[idx]
        if n > 2:
            g = np.diff(t)
            f['gap_mod5'] = float((g % 5 == 0).mean())
            f['gap_mod10'] = float((g % 10 == 0).mean())
            f['gap_mod60'] = float((g % 60 == 0).mean())
            f['gap_nuniq_rate'] = float(len(np.unique(g)) / len(g))
        else:
            f['gap_mod5'] = f['gap_mod10'] = f['gap_mod60'] = f['gap_nuniq_rate'] = 0.0

        ii = item[idx][~np.isnan(item[idx])]
        if len(ii) > 1:
            di = np.diff(ii)
            f['item_mono'] = float(((di > 0).mean() + (di < 0).mean()))
            f['item_med_step'] = float(np.median(np.abs(di)))
            f['item_revisit'] = float(len(ii) - len(np.unique(ii)))
        else:
            f['item_mono'] = f['item_med_step'] = f['item_revisit'] = 0.0

        e = nm[idx]
        f['no_login_no_fav'] = int('login' not in e and 'favorite_add' not in e)
        f['contact_never'] = int(not any(x.startswith('contact') for x in e))
        f['n_kinds_live'] = int(sum(x in e for x in
                                    ['login', 'favorite_add', 'contact_chat_open',
                                     'contact_message_sent']))
        if n > 3:
            f['seq_search_item'] = float(sum(a == 'search_results_view' and b == 'item_view'
                                             for a, b in zip(e[:-1], e[1:])) / n)
            f['seq_view_swipe'] = float(sum(a == 'item_view' and b == 'photo_swipe'
                                            for a, b in zip(e[:-1], e[1:])) / n)
            f['seq_seller_then_contact'] = float(
                sum(a == 'seller_page_view' and b.startswith('contact')
                    for a, b in zip(e[:-1], e[1:])) / n)
        else:
            f['seq_search_item'] = f['seq_view_swipe'] = f['seq_seller_then_contact'] = 0.0

        h = pd.to_datetime(t, unit='s').hour.values
        f['hour_range'] = int(h.max() - h.min()) if n else 0
        f['first_hour_share'] = float((h == h[0]).mean()) if n else 0.0
        f['night_only'] = int(bool(((h >= 0) & (h < 6)).all())) if n else 0
        f['day_only'] = int(bool(((h >= 9) & (h < 19)).all())) if n else 0
        f['n_distinct_hours_ratio'] = len(np.unique(h)) / max(n, 1)
        out[cid] = f

    df = pd.DataFrame.from_dict(out, orient='index')
    df.index.name = 'cookie_id'
    return df.reset_index().fillna(0.0).replace([np.inf, -np.inf], 0.0)

### 3.2 Доменные признаки (без меток)

Здесь считаем популярность среди **других кук того же дня** — это «плейсхолдер»
контекста без утечки меток: мы не знаем, бот или человек сосед, но знаем, что объект
смотрели все или только мы. Логика такая:

* человек часто ходит по популярным объявлениям и «свежим» запросам из подсказок;
* бот-скрейпер тянет редкие/узкие хвосты, у него средняя популярность ниже;
* дополнительно: смена регистра платформы подряд (признак «режим переключили»),
  прыжки между городами (человек редко перелетает между регионами за секунды),
  наличие курсора по типам событий (у браузерной автоматизации указатель
  либо всегда есть, либо всегда нет), длина «залпа» одинаковых событий подряд.

Всё считается **без меток** — только по событиям этого же дня.

In [23]:
def day_global_features(w: pd.DataFrame) -> pd.DataFrame:
    w = w.copy()
    w['day'] = w.event_ts.values.astype('datetime64[D]')

    item_day = w.dropna(subset=['item_id']).groupby(['day', 'item_id']).cookie_id.nunique()
    q_day = w.dropna(subset=['search_query']).groupby(['day', 'search_query']).cookie_id.nunique()
    cat_day = w.dropna(subset=['item_category']).groupby(['day', 'item_category']).cookie_id.nunique()
    loc_day = w.dropna(subset=['item_location']).groupby(['day', 'item_location']).cookie_id.nunique()

    w = w.merge(item_day.rename('item_pop').reset_index(), on=['day', 'item_id'], how='left')
    w = w.merge(q_day.rename('q_pop').reset_index(), on=['day', 'search_query'], how='left')
    w = w.merge(cat_day.rename('cat_pop').reset_index(), on=['day', 'item_category'], how='left')
    w = w.merge(loc_day.rename('loc_pop').reset_index(), on=['day', 'item_location'], how='left')

    g = w.groupby('cookie_id', sort=False)
    items = w.dropna(subset=['item_id'])
    queries = w.dropna(subset=['search_query'])
    f = pd.DataFrame({
        'item_pop_mean': g.item_pop.mean(),
        'item_pop_max': g.item_pop.max(),
        'item_rare_share': items.groupby('cookie_id', sort=False).item_pop
                                   .apply(lambda s: (s.fillna(1) <= 1).mean()),
        'q_pop_mean': g.q_pop.mean(),
        'q_rare_share': queries.groupby('cookie_id', sort=False).q_pop
                                   .apply(lambda s: (s.fillna(1) <= 1).mean()),
        'cat_pop_mean': g.cat_pop.mean(),
        'loc_pop_mean': g.loc_pop.mean(),
        'loc_pop_max': g.loc_pop.max(),
    })

    pid = w.groupby('cookie_id', sort=False).indices
    plat, loc = w.platform.values, w.item_location.values
    nm_all = w.event_name.values
    ts_all = w.event_ts.values.astype('datetime64[s]').astype(np.int64)
    chg, case_ent, loc_chg, same_gap, run_max = {}, {}, {}, {}, {}
    for cid, idx in pid.items():
        pl = plat[idx]
        if len(pl) > 1:
            chg[cid] = float((pl[1:] != pl[:-1]).mean())
        else:
            chg[cid] = 0.0
        _, c = np.unique(pl, return_counts=True)
        pv = c / c.sum()
        case_ent[cid] = float(-(pv * np.log(pv)).sum())

        l = loc[idx]
        l = l[~pd.isna(l)]
        loc_chg[cid] = float((l[1:] != l[:-1]).mean()) if len(l) > 1 else 0.0

        e = nm_all[idx]
        t = ts_all[idx]
        gaps = [t[k + 1] - t[k] for k in range(len(e) - 1) if e[k] == e[k + 1]]
        same_gap[cid] = float(np.mean(gaps)) if gaps else 0.0
        if len(e) > 1:
            cur = best = 1
            for a, b in zip(e[:-1], e[1:]):
                cur = cur + 1 if a == b else 1
                best = max(best, cur)
            run_max[cid] = best
        else:
            run_max[cid] = 1

    f['plat_change_rate'] = pd.Series(chg)
    f['plat_case_entropy'] = pd.Series(case_ent)
    f['loc_change_rate'] = pd.Series(loc_chg)
    f['same_type_gap'] = pd.Series(same_gap)
    f['max_same_run'] = pd.Series(run_max)
    f['ptr_on_item'] = w[w.event_name == 'item_view'].groupby('cookie_id', sort=False).pointer_x \
                                                     .apply(lambda s: s.notna().mean())
    f['ptr_on_search'] = w[w.event_name == 'search_results_view'].groupby('cookie_id', sort=False) \
                                                     .pointer_x.apply(lambda s: s.notna().mean())
    return f.rename_axis('cookie_id').reset_index().fillna(0.0).replace([np.inf, -np.inf], 0.0)

### 3.3 Среда использования

Чем «окружающая среда» куки отличается у автоматизации:

* **микс платформ** — реальный пользователь почти всегда сидит на одной платформе;
  боты чаще крутят несколько сразу или сидят на десктопном вебе без мобильных;
* **микс семейств UA** — то же самое на уровне браузера: `HeadlessChrome` и резкие
  переключения между семействами выдают скрипт;
* **доля `pro`-продавцов** — поток, нацеленный на коммерцию, отличается от обычного
  пользовательского просмотра;
* **топовые доли** (категория/локация/запрос/товар) — «сфокусированный» трафик
  крутит почти один объект (скрейп одного раздела), человек размазан шире;
* **число пар** «категория × локация» и «платформа × UA» — компактность следа.

In [25]:
def env_mix_features(w: pd.DataFrame) -> pd.DataFrame:
    w = w.copy()
    w['plat_l'] = w.platform.str.lower().replace({'iphone': 'ios'})
    w['uak'] = w.user_agent.map(ua_kind)
    g = w.groupby('cookie_id', sort=False)
    out = pd.DataFrame(index=g.size().index)

    pm = w.pivot_table(index='cookie_id', columns='plat_l', values='eid',
                       aggfunc='count').reindex(out.index).fillna(0)
    pm = pm.reindex(columns=PLAT_LEVELS, fill_value=0)
    for c in PLAT_LEVELS:
        out[f'pshare_{c}'] = pm[c]

    um = w.pivot_table(index='cookie_id', columns='uak', values='eid',
                       aggfunc='count').reindex(out.index).fillna(0)
    um = um.reindex(columns=UA_KINDS, fill_value=0)
    for c in UA_KINDS:
        out[f'ushare_{c}'] = um[c]
    out['n_ua_kinds'] = (um > 0).sum(1)

    out['share_pro'] = w.seller_type.eq('pro').groupby(w.cookie_id, sort=False).mean()
    out['share_private'] = w.seller_type.eq('private').groupby(w.cookie_id, sort=False).mean()

    def top_share(col):
        s = w.dropna(subset=[col])
        vc = s.groupby('cookie_id')[col].value_counts()
        mx = vc.groupby('cookie_id').max()
        tot = vc.groupby('cookie_id').sum()
        return (mx / tot).rename(f'top_{col}_share')

    out['top_cat_share'] = top_share('item_category')
    out['top_loc_share'] = top_share('item_location')
    out['top_query_share'] = top_share('search_query')
    out['top_item_share'] = top_share('item_id')

    pairs = w.dropna(subset=['item_category', 'item_location']) \
             [['cookie_id', 'item_category', 'item_location']].drop_duplicates()
    out['n_cat_loc_pairs'] = pairs.groupby('cookie_id').size()
    pu = w[['cookie_id', 'plat_l', 'uak']].drop_duplicates()
    out['n_plat_ua_pairs'] = pu.groupby('cookie_id').size()

    return out.replace([np.inf, -np.inf], 0.0).reset_index().fillna(0.0)

### 3.4 Дневные перцентили

Одно наблюдение из осмотра данных: счётчики сильно зависят от **длины окна** — в один
день все куки активнее, в другой тише, поэтому сырые значения между днями несравнимы.
Решение: для ключевых признаков внутри каждого суточного дня считаем **перцентиль**
(«кука в таком-то проценте кук этого дня») и **z-оценку** («на сколько сигм выше
среднего дня»). Метки не используются, привязка только к своему дню — значит, признак
честно доступен к `window_end_ts` и ничего не течёт из будущего.

In [27]:
def add_day_relative(X: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Перцентили и z-оценки внутри того же суточного окна (метки не используются)."""
    day = pd.Series(meta.window_start_ts.values.astype('datetime64[D]'))
    keys = ['n_events', 'gap_med', 'gap_cv', 'span_frac', 'items_per_evt', 'cats_per_evt',
            'locs_per_evt', 'page_max', 'n_items', 'cookie_age_days', 'hour_mean',
            'evt_entropy', 'n_sessions', 'r_item_view', 'r_photo_swipe', 't_first', 't_last',
            'dup_row_rate', 'n_queries', 'query_entropy',
            'hour_range', 'no_login_no_fav', 'gap_nuniq_rate', 'n_distinct_hours_ratio']
    out = X.copy()
    for col in keys:
        if col not in out.columns:
            continue
        g = out[col].groupby(day)
        out[f'{col}_pct'] = g.rank(pct=True, method='average')
        mu, sd = g.transform('mean'), g.transform('std')
        out[f'{col}_z'] = ((out[col] - mu) / sd.replace(0, np.nan)).fillna(0.0)
    return out

### 3.5 Сборка матрицы

Слои склеиваем **по порядку, отдельно для train и test**, печатая форму после каждого —
так видно, какой слой что добавил (и проще ловить ошибки: например, рассинхрон колонок).
В конце — дневные перцентили, финальная заливка пропусков и проверки.

In [29]:
# слой 1: ядро (состав потока, тайминги, сессии, курсор, UA, переходы, дубликаты)
X_train = core_features(ev_tr, train)
X_test = core_features(ev_te, test)
print('ядро:', X_train.shape, '| тест:', X_test.shape, '|', elapsed())

ядро: (11091, 135) | тест: (4909, 135) | 69s


In [30]:
# слой 2: скриптовое поведение (пагинация, круглые паузы, отсутствие «живых» действий)
X_train = X_train.merge(script_features(ev_tr), on='cookie_id', how='left')
X_test = X_test.merge(script_features(ev_te), on='cookie_id', how='left')
print('+ скриптовое поведение:', X_train.shape, '|', elapsed())

+ скриптовое поведение: (11091, 159) | 79s


In [31]:
# слой 3: доменные признаки (популярность объектов внутри дня, смена регистра, прыжки локаций)
X_train = X_train.merge(day_global_features(ev_tr), on='cookie_id', how='left')
X_test = X_test.merge(day_global_features(ev_te), on='cookie_id', how='left')
print('+ доменные:', X_train.shape, '|', elapsed())

+ доменные: (11091, 174) | 91s


In [32]:
# слой 4: среда использования (микс платформ/UA, pro-продавцы, топовые доли)
X_train = X_train.merge(env_mix_features(ev_tr), on='cookie_id', how='left')
X_test = X_test.merge(env_mix_features(ev_te), on='cookie_id', how='left')
print('+ среда:', X_train.shape, '|', elapsed())

+ среда: (11091, 193) | 93s


In [33]:
# слой 5: перцентили и z-оценки внутри своего суточного дня (метки не используются)
X_train = add_day_relative(X_train, train)
X_test = add_day_relative(X_test, test)

X_train = X_train.fillna(0.0).replace([np.inf, -np.inf], 0.0)
X_test = X_test.fillna(0.0).replace([np.inf, -np.inf], 0.0)

assert list(X_train.cookie_id) == list(train.cookie_id), 'порядок кук train изменился'
assert list(X_test.cookie_id) == list(test.cookie_id), 'порядок кук test изменился'
assert list(X_train.columns) == list(X_test.columns), 'колонки train/test разошлись'

print('итог: train', X_train.shape, '| test', X_test.shape,
      '| колонки совпадают: True |', elapsed())
X_train.head(3)

итог: train (11091, 241) | test (4909, 241) | колонки совпадают: True | 93s


,cookie_id,n_events,n_item_view,r_item_view,n_search_results_view,r_search_results_view,n_photo_swipe,r_photo_swipe,n_favorite_add,r_favorite_add,...,query_entropy_pct,query_entropy_z,hour_range_pct,hour_range_z,no_login_no_fav_pct,no_login_no_fav_z,gap_nuniq_rate_pct,gap_nuniq_rate_z,n_distinct_hours_ratio_pct,n_distinct_hours_ratio_z
0,ck_54a059eb7d3ea68b,7,3,0.428571,3,0.428571,1,0.142857,0,0.000000,...,0.539508,0.119418,0.122409,-0.952000,0.813472,1.291942,0.734456,0.501671,0.250000,-0.637631
1,ck_7e4de46eeab82974,41,22,0.536585,8,0.195122,4,0.097561,1,0.024390,...,0.730570,0.748671,0.964378,2.384518,0.313472,-0.773026,0.309585,0.116128,0.436528,-0.369910
2,ck_9320229ef6304522,36,7,0.194444,13,0.361111,3,0.083333,3,0.083333,...,0.955311,1.629417,0.949482,2.081198,0.313472,-0.773026,0.141192,-0.379571,0.435233,-0.373381


### 3.6 Категориальные признаки

Отдельный слой из **категорий** — сырых значений полей событий окна, а не агрегатов:
семейство UA, набор и доминанта платформ, доминанта/первое/последнее событие,
топ-категория/локация/товар/запрос, набор `seller_type`, пиковый час, бакет объёма.
Нужен для CatBoost (`cat_features`) и LightGBM (`category`-dtype): числовые агрегаты
не несут информации о **идентичности** значений (какой именно товар/запрос в топе).
Важно: одни категории сами по себе почти бесполезны (проверено ниже: 0.13), но в
блендинге с числовыми моделями дают заметный прирост — разные модели начинают
смотреть на разные аспекты поведения.

In [35]:
def mode_str(s, cast=str):
    """Мода колонки как строка; пустые значения → 'NA'."""
    s = s.dropna()
    if len(s) == 0:
        return 'NA'
    return cast(s.value_counts().index[0])


def make_cats(ev: pd.DataFrame, meta: pd.DataFrame) -> pd.DataFrame:
    """Категориальные признаки куки только по событиям внутри окна."""
    ids = list(meta.cookie_id)
    w = ev[['cookie_id', 'event_ts', 'event_name', 'platform', 'user_agent',
            'item_id', 'item_category', 'item_location', 'search_query', 'seller_type']].copy()
    w['plat_l'] = w['platform'].str.lower().replace({'iphone': 'ios'})
    g = w.groupby('cookie_id', sort=False)

    df = pd.DataFrame({'cookie_id': ids}).set_index('cookie_id')
    df['ua_kind'] = g['user_agent'].first().reindex(df.index) \
        .map(lambda u: ua_kind(u) if isinstance(u, str) else 'none')
    df['ua_nuniq'] = g['user_agent'].nunique().reindex(df.index).fillna(0).astype(int).astype(str)
    df['plat_set'] = g['plat_l'].agg(lambda s: ','.join(sorted(set(s)))).reindex(df.index)
    df['plat_case_set'] = g['platform'].agg(lambda s: ','.join(sorted(set(s)))).reindex(df.index)
    df['dom_plat'] = g['plat_l'].agg(mode_str).reindex(df.index)
    df['dom_evt'] = g['event_name'].agg(mode_str).reindex(df.index)
    df['first_evt'] = g['event_name'].first().reindex(df.index)
    df['last_evt'] = g['event_name'].last().reindex(df.index)
    df['top_cat'] = g['item_category'].agg(mode_str).reindex(df.index)
    df['top_loc'] = g['item_location'].agg(mode_str).reindex(df.index)
    df['top_item'] = g['item_id'].agg(mode_str).reindex(df.index)
    df['top_query'] = g['search_query'].agg(mode_str).reindex(df.index)
    df['seller_set'] = g['seller_type'].agg(
        lambda s: ','.join(sorted(set(s.dropna()))) or 'NA').reindex(df.index)
    df['peak_hour'] = g['event_ts'].agg(
        lambda s: str(int(s.dt.hour.value_counts().index[0]))).reindex(df.index)
    n = g.size().reindex(df.index)
    df['n_bucket'] = pd.cut(n, [-1, 0, 2, 5, 10, 20, 40, 80, 200, 10 ** 9],
                            labels=['0', '1-2', '3-5', '6-10', '11-20',
                                    '21-40', '41-80', '81-200', '200+']).astype(str)
    df = df.reset_index().fillna('NA')
    assert list(df.cookie_id) == list(meta.cookie_id)
    return df

In [36]:
CT_tr = make_cats(ev_tr, train)
CT_te = make_cats(ev_te, test)
CATS = [c for c in CT_tr.columns if c != 'cookie_id']
CT_tr = CT_tr.reset_index(drop=True)
CT_te = CT_te.reset_index(drop=True)
print('категориальных фич:', len(CATS))
print('cardinality:', {c: int(CT_tr[c].nunique()) for c in CATS})
CT_tr.head(3)

категориальных фич: 15
cardinality: {'ua_kind': 6, 'ua_nuniq': 3, 'plat_set': 5, 'plat_case_set': 37, 'dom_plat': 4, 'dom_evt': 9, 'first_evt': 9, 'last_evt': 9, 'top_cat': 16, 'top_loc': 41, 'top_item': 9840, 'top_query': 121, 'seller_set': 4, 'peak_hour': 24, 'n_bucket': 7}


,cookie_id,ua_kind,ua_nuniq,plat_set,plat_case_set,dom_plat,dom_evt,first_evt,last_evt,top_cat,top_loc,top_item,top_query,seller_set,peak_hour,n_bucket
0,ck_54a059eb7d3ea68b,other,1,android,"ANDROID,Android,android",android,search_results_view,search_results_view,item_view,kvartiry_prodazha,sankt-peterburg,1028802.0,купить 1к квартиру,"private,pro",1,6-10
1,ck_7e4de46eeab82974,yabrowser,1,"desktop,web","WEB,Web,desktop,web",web,item_view,photo_swipe,search_results_view,kvartiry_arenda,vladivostok,1042434.0,снять квартиру у метро,"private,pro",10,41-80
2,ck_9320229ef6304522,chrome,1,"desktop,web","WEB,Web,desktop,web",web,search_results_view,search_results_view,contact_phone_show,detskie_tovary,novosibirsk,1035992.0,посудомоечная машина,"private,pro",1,21-40


Теперь собираем три рабочие матрицы:

* `XNUM_*` — только числа (240 колонок): для чисто числовых LightGBM/CatBoost;
* `XCAT_*` — только категории (строки): для отдельной проверочной модели CatBoost;
* `XMIX_*` — числа + категории в `dtype=category`: для микс-моделей.

In [38]:
FEATS = [c for c in X_train.columns if c != 'cookie_id']   # числовые признаки
XNUM_tr = X_train[FEATS].reset_index(drop=True)
XNUM_te = X_test[FEATS].reset_index(drop=True)
XCAT_tr = CT_tr[CATS].astype(str)
XCAT_te = CT_te[CATS].astype(str)
# смешанная матрица: числовые признаки + категории в dtype category
XMIX_tr = pd.concat([XNUM_tr, XCAT_tr.astype('category')], axis=1)
XMIX_te = pd.concat([XNUM_te, XCAT_te.astype('category')], axis=1)
MIX_FEATS = list(XMIX_tr.columns)
print('матрицы: XNUM', XNUM_tr.shape, '| XMIX', XMIX_tr.shape, '|', elapsed())

матрицы: XNUM (11091, 240) | XMIX (11091, 255) | 129s


## 4. Диагностика признаков

Прежде чем обучать модели, проверяем признаки **поодиночке**: считаем AUC каждой колонки
(и берём `max(AUC, 1-AUC)`, чтобы ловить и инвертированные признаки). Это отвечает на
два вопроса сразу:

* есть ли в наборе признаки, которые почти ничего не разделяют (AUC ≈ 0.5) — шум,
  который модель придётся отфильтровывать самой;
* какие признаки работают **сами по себе** — обычно это разнообразие локаций/категорий,
  глубина пагинации, темп и регулярность потока, возраст куки, доля повторных запросов.

Важно: одномерный AUC — только ориентир. Настоящий отбор делает модель на валидации,
а не этот список (поэтому мы и не выкидываем «слабые» признаки: в комбинации они
часто дают больше, чем поодиночке).

In [40]:
from sklearn.metrics import roc_auc_score

y = train.target.values
FEATS = [c for c in X_train.columns if c != 'cookie_id']

uni = {}
for c in FEATS:
    v = X_train[c].values
    if np.unique(v).size < 2:
        continue
    uni[c] = max(roc_auc_score(y, v), roc_auc_score(y, -v))
uni = pd.Series(uni).sort_values(ascending=False)
print('признаков с AUC > 0.65:', int((uni > 0.65).sum()))
print('признаков с AUC < 0.55:', int((uni < 0.55).sum()))
uni.head(20).round(3)

признаков с AUC > 0.65: 46
признаков с AUC < 0.55: 98


events_per_minute             0.727
cats_per_evt_z                0.721
gap_lt60                      0.721
cats_per_evt_pct              0.721
cats_per_evt                  0.721
gap_q75                       0.720
loc_entropy                   0.714
n_locs                        0.711
page_mean                     0.710
gap_med                       0.709
gap_med_pct                   0.709
page_max_z                    0.705
page_max_pct                  0.705
page_max                      0.705
query_repeat                  0.701
page_std                      0.698
gap_med_z                     0.696
n_distinct_hours_ratio_z      0.695
n_distinct_hours_ratio_pct    0.695
events_per_active_hour        0.695
dtype: float64

То, что почти все признаки выше 0.55, — хороший знак: набор не состоит из шума.
Слабые поодиночке признаки оставляем: деревья используют их в комбинациях.

## 5. Валидация

Тест лежит позже трейна, поэтому случайный сплит врёт: он даёт будущие дни в
обучении и прошлые в проверке. Используем walk-forward: для каждого из последних
7 дней (04-13…04-19) обучаемся на всех более ранних днях и предсказываем этот день,склеивая предсказания семи дней, получаем out-of-fold скоры для ~5.2k кук — в 7 раз
больше ботов, чем в одном сплите, — и считаем метрику официальным `metric.py`

Правила, которых держимся:

- сравниваем модели не по одному числу, а по **полному P@R + среднему по дням** - разброс по дням ±0.05–0.07, один «удачный» день ничего не решает;
- порог не подбираем — его подбирает проверяющая система.

Сравниваем подходы по порядку: baseline из quickstart → деревья на всех признаках →
LightGBM (4 конфигурации сложности) → CatBoost → микс-модели → ансамбль.

In [43]:
day = train.window_start_ts.values.astype('datetime64[D]')
VALID_DAYS = np.unique(day)[-7:]
print('дни валидации:', [str(d) for d in VALID_DAYS])
print('всего дней в train:', len(np.unique(day)),
      '| кук в валидации:', int(np.isin(day, VALID_DAYS).sum()))


def walk_forward(fit_predict, feats=FEATS, mat=None, **kw):
    """Обучение на прошлых днях, предсказание следующего дня.

    `mat` — матрица признаков (по умолчанию X_train); для микс-моделей передаём XMIX_tr.
    """
    M = X_train if mat is None else mat
    pred = np.full(len(y), np.nan)
    for d in VALID_DAYS:
        tr_idx = np.where(day < d)[0]
        va_idx = np.where(day == d)[0]
        pred[va_idx] = fit_predict(M.iloc[tr_idx][feats], y[tr_idx],
                                   M.iloc[va_idx][feats], **kw)
    return pred


def report(pred, label):
    m = ~np.isnan(pred)
    sub_y, sub_p = y[m], pred[m]
    p = precision_at_recall(sub_y, sub_p)
    per_day = [precision_at_recall(y[m & (day == d)], pred[m & (day == d)])
               for d in VALID_DAYS if y[m & (day == d)].sum() > 0]
    print(f'{label:34s} P@R0.7={p:.4f}  среднее по дням={np.mean(per_day):.4f}  '
          f'n={m.sum()} ботов={sub_y.sum()}  {elapsed()}')
    return p

дни валидации: ['2026-04-13', '2026-04-14', '2026-04-15', '2026-04-16', '2026-04-17', '2026-04-18', '2026-04-19']
всего дней в train: 14 | кук в валидации: 5161


### 5.1 Baseline и деревья

Начинаем с простого, чтобы было с чем сравнивать:

* **baseline из quickstart** — ExtraTrees всего на двух признаках (`n_events`, `n_items`).
  Он отвечает на вопрос «насколько вообще признаки важнее модели»: если он даёт ~0.10,
  а весь набор — ~0.7, значит вся сила решения в признаках;
* **ExtraTrees на всех ~240 признаках** — уже серьёзный прыжок, но деревья хуже
  LightGBM работают с сотнями слабокоррелированных колонок и шумом.

In [45]:
from sklearn.ensemble import ExtraTreesClassifier


def et_predict(X, yy, Xv, seed=RANDOM_STATE, **kw):
    p = dict(n_estimators=500, min_samples_leaf=5, max_features='sqrt',
             class_weight='balanced', random_state=seed, n_jobs=-1)
    p.update(kw)
    m = ExtraTreesClassifier(**p)
    m.fit(X, yy)
    return m.predict_proba(Xv)[:, 1]


# baseline quickstart: два агрегата
simple_cols = ['n_events', 'n_items']
oof_rf_simple = walk_forward(lambda X, yy, Xv: et_predict(X, yy, Xv, n_estimators=300),
                             feats=simple_cols)
report(oof_rf_simple, 'baseline: 2 признака (quickstart)')

oof_rf = walk_forward(et_predict)
report(oof_rf, 'ExtraTrees: все признаки')

baseline: 2 признака (quickstart)  P@R0.7=0.1093  среднее по дням=0.1116  n=5161 ботов=408  136s
ExtraTrees: все признаки           P@R0.7=0.5934  среднее по дням=0.6195  n=5161 ботов=408  155s


0.5933609958506224

### 5.2 LightGBM: четыре конфигурации

Гоняем четыре конфигурации разной «глубины» — от узкой (`leaves15`, минимум сложности)
до длинной (`deep`, много деревьев с маленьким шагом). Их ценность не в поодиночном
рекорде, а в том, что они ошибаются чуть по-разному, поэтому в ансамбле дают больше,
чем лучшая из них одна. (Проверено: конфиг `small` с 400 деревьями даёт самый высокий
отдельный OOF, а `leaves15` — самый низкий, но все четверо остаются в ансамбле.)

In [47]:
def lgb_predict(X, yy, Xv, seed=RANDOM_STATE, **kw):
    import lightgbm as lgb
    p = dict(objective='binary', metric='auc', verbosity=-1, random_state=seed,
             n_estimators=800, learning_rate=0.03, num_leaves=63, min_child_samples=30,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
             reg_alpha=0.1, reg_lambda=1.0)
    p.update(kw)
    m = lgb.LGBMClassifier(**p)
    m.fit(X, yy)
    return m.predict_proba(Xv)[:, 1]


LGB_CONFIGS = {
    'default': dict(n_estimators=800, learning_rate=0.03, num_leaves=63, min_child_samples=30),
    'deep': dict(n_estimators=1500, learning_rate=0.02, num_leaves=63, min_child_samples=20),
    'leaves15': dict(num_leaves=15, min_child_samples=50),
    'small': dict(n_estimators=400, learning_rate=0.05),
}

oof_lgb = {}
for name, cfg in LGB_CONFIGS.items():
    oof_lgb[name] = walk_forward(lgb_predict, **cfg)
    report(oof_lgb[name], f'LightGBM: {name}')

LightGBM: default                  P@R0.7=0.7077  среднее по дням=0.7232  n=5161 ботов=408  213s
LightGBM: deep                     P@R0.7=0.7352  среднее по дням=0.7256  n=5161 ботов=408  310s
LightGBM: leaves15                 P@R0.7=0.7132  среднее по дням=0.7335  n=5161 ботов=408  334s
LightGBM: small                    P@R0.7=0.7079  среднее по дням=0.7225  n=5161 ботов=408  359s


### 5.3 CatBoost на числовых признаках

CatBoost — третья «одиночная» модель числового семейства: другие деревья (обучение
с бустингом по схеме от GBM, симметричные деревья, другой регуляризатор), поэтому
ошибается иначе, чем LightGBM. Для числовых признаков он примерно на уровне лучшего
LightGBM, зато в микс-варианте (секция 5.4) добавляет в ансамбль категориальный взгляд.

In [49]:
def cb_predict(X, yy, Xv, seed=RANDOM_STATE, cat_names=None, **kw):
    from catboost import CatBoostClassifier
    p = dict(iterations=500, depth=6, learning_rate=0.05, l2_leaf_reg=8,
             random_seed=seed, verbose=False, auto_class_weights='Balanced')
    p.update(kw)
    idx = [list(X.columns).index(c) for c in cat_names] if cat_names else None
    m = CatBoostClassifier(**p)
    m.fit(X, yy, cat_features=idx)
    return m.predict_proba(Xv)[:, 1]


oof_cb = walk_forward(cb_predict)
report(oof_cb, 'CatBoost: 500 итераций')

CatBoost: 500 итераций             P@R0.7=0.7114  среднее по дням=0.6859  n=5161 ботов=408  453s


0.7114427860696517

### 5.4 Микс-модели (числовые + категориальные)

Те же алгоритмы, но на смешанной матрице: LightGBM принимает категории как
`category`-dtype, CatBoost — через `cat_features`. Поодиночке эти модели **не лучше**
числовых (LGBM даже слабее), зато ошибаются иначе: Spearman-корреляция микс-моделей
с числовыми ~0.81 против 0.98 внутри числового семейства, поэтому в блендинге они
дают настоящий выигрыш. Один CatBoost на одних категориях, без числовых, дал бы ~0.13 —
категории сами по себе бесполезны, работает только связка «категории + числа».

In [51]:
oof_mix = {}
oof_mix['lgb_mix'] = walk_forward(lgb_predict, feats=MIX_FEATS, mat=XMIX_tr)
report(oof_mix['lgb_mix'], 'LGBM микс (числ+кат)')

oof_mix['lgb_mix_deep'] = walk_forward(lgb_predict, feats=MIX_FEATS, mat=XMIX_tr,
                                       n_estimators=1500, learning_rate=0.02,
                                       min_child_samples=20)
report(oof_mix['lgb_mix_deep'], 'LGBM микс deep')

oof_mix['cb_mix'] = walk_forward(cb_predict, feats=MIX_FEATS, mat=XMIX_tr, cat_names=CATS)
report(oof_mix['cb_mix'], 'CatBoost микс (числ+кат)')

# в ансамбль не входит — нужен только как доказательство тезиса о категориях
oof_cb_cat = walk_forward(cb_predict, feats=CATS, mat=XCAT_tr, cat_names=CATS)
report(oof_cb_cat, 'CatBoost: только категории (в ансамбль не идёт)')

LGBM микс (числ+кат)               P@R0.7=0.6809  среднее по дням=0.6957  n=5161 ботов=408  499s
LGBM микс deep                     P@R0.7=0.6892  среднее по дням=0.7182  n=5161 ботов=408  580s
CatBoost микс (числ+кат)           P@R0.7=0.6859  среднее по дням=0.7125  n=5161 ботов=408  847s
CatBoost: только категории (в ансамбль не идёт) P@R0.7=0.1308  среднее по дням=0.1392  n=5161 ботов=408  1105s


0.13075170842824602

### 5.5 Ансамбль

Усредняем **процентили** скоров, а не сырые: у моделей разная калибровка (одни дают
больше 0.9, другие жмутся к 0.5), а метрике важен **только порядок кук**. Перевод
каждый скор в его процентиль внутри выборки делает модели сравнимыми между собой.

Веса групповые: числовое семейство (4 конфигурации LightGBM) — по **1.0**,
микс-тройка (LGBM default, LGBM deep, CatBoost) — по **0.5**. Микс-модели полезны,
но каждая поодиночке слабее числовых, и перевес 1:1 их «перетягивает»: ансамбль
начинает хуже ранжировать. Вес подобран на walk-forward по полному P@R и среднему
по дням: плато на 0.45–0.55, оба критерия лучшие в этой области.

Ниже сравниваем четыре варианта комбинации и смотрим метрику по дням.

In [53]:
def rank_pct(a):
    return pd.Series(a).rank(pct=True).values


def blend(scores: dict, weights: dict) -> np.ndarray:
    """Взвешенное среднее процентилей скоров."""
    keys = list(scores)
    st = np.vstack([scores[k] for k in keys])
    w = np.array([weights[k] for k in keys], dtype=float)
    w = w / w.sum()
    out = (np.vstack([rank_pct(row) for row in st]) * w[:, None]).sum(axis=0)
    out[np.isnan(st).any(axis=0)] = np.nan
    return out


oof_all = {**oof_lgb, **oof_mix}
W_NUM, W_MIX = 1.0, 0.5
WEIGHTS = {**{k: W_NUM for k in LGB_CONFIGS},
           'lgb_mix': W_MIX, 'lgb_mix_deep': W_MIX, 'cb_mix': W_MIX}

oof_ens4 = blend({k: oof_lgb[k] for k in LGB_CONFIGS},
                 {k: 1.0 for k in LGB_CONFIGS})
report(oof_ens4, 'ансамбль 4x LGBM (числовые)')

oof_ens = blend(oof_all, WEIGHTS)
report(oof_ens, 'ансамбль 7 моделей (числ + микс)')

print('\nсравнение блендингов:')
blend_variants = {
    'ens4 (только числовые)': ({k: oof_lgb[k] for k in LGB_CONFIGS},
                               {k: 1.0 for k in LGB_CONFIGS}),
    'mix3 (только микс)': (oof_mix, {k: 1.0 for k in oof_mix}),
    'ens4 + cb_mix + lgb_mix_deep': ({**{k: oof_lgb[k] for k in LGB_CONFIGS},
                                      'cb_mix': oof_mix['cb_mix'],
                                      'lgb_mix_deep': oof_mix['lgb_mix_deep']},
                                     {**{k: 1.0 for k in LGB_CONFIGS},
                                      'cb_mix': 1.0, 'lgb_mix_deep': 1.0}),
    'ens4 + микс 0.5 (финал)': (oof_all, WEIGHTS),
}
for label, (sc, wt) in blend_variants.items():
    report(blend(sc, wt), label)

print('\nметрика по дням для финального ансамбля:')
rows = []
for d in VALID_DAYS:
    m = (~np.isnan(oof_ens)) & (day == d)
    rows.append({'день': str(np.datetime64(d, 'D')), 'куки': int(m.sum()),
                 'ботов': int(y[m].sum()),
                 'P@R0.7': round(precision_at_recall(y[m], oof_ens[m]), 3)})
print(pd.DataFrame(rows).to_string(index=False))

ансамбль 4x LGBM (числовые)        P@R0.7=0.7359  среднее по дням=0.7248  n=5161 ботов=408  1105s
ансамбль 7 моделей (числ + микс)   P@R0.7=0.7409  среднее по дням=0.7315  n=5161 ботов=408  1105s

сравнение блендингов:
ens4 (только числовые)             P@R0.7=0.7359  среднее по дням=0.7248  n=5161 ботов=408  1105s
mix3 (только микс)                 P@R0.7=0.7073  среднее по дням=0.7288  n=5161 ботов=408  1105s
ens4 + cb_mix + lgb_mix_deep       P@R0.7=0.7409  среднее по дням=0.7331  n=5161 ботов=408  1105s
ens4 + микс 0.5 (финал)            P@R0.7=0.7409  среднее по дням=0.7315  n=5161 ботов=408  1105s

метрика по дням для финального ансамбля:
      день  куки  ботов  P@R0.7
2026-04-13   843     53   0.776
2026-04-14   826     70   0.721
2026-04-15   801     65   0.730
2026-04-16   740     60   0.694
2026-04-17   713     66   0.783
2026-04-18   606     52   0.650
2026-04-19   632     42   0.767


## 6. Финальная модель и сабмит

Walk-forward закончился — теперь переносим **точно ту же процедуру** на весь train:

1. обучаем те же 4 числовых LightGBM на всех ~16k кук (не только на 7 днях валидации);
2. обучаем микс-тройку (2 LightGBM + CatBoost) на матрице «числа + категории»;
3. собираем скор теста взвешенным средним **процентилей** с теми же весами 1.0/0.5;
4. порог не подбираем — его подбирает проверяющая система, мы даём непрерывный score.

Отдельно смотрим, насколько каждый скор на тесте коррелирует с финальным ансамблем:
ожидаем ~0.8–0.95 — отдельные модели сходятся с ансамблем, но не дублируют его.

In [55]:
# числовое семейство: 4 конфигурации LightGBM на всём train
test_raw = {}
for name, cfg in LGB_CONFIGS.items():
    test_raw[name] = lgb_predict(X_train[FEATS], y, X_test[FEATS], **cfg)
    print(f'{name:10s} готов | {elapsed()}')

default    готов | 1114s
deep       готов | 1129s
leaves15   готов | 1133s
small      готов | 1137s


Микс-тройка: те же алгоритмы, но на матрице с категориальными колонками
(`category`-dtype для LightGBM, `cat_features` для CatBoost).

In [57]:
test_raw['lgb_mix'] = lgb_predict(XMIX_tr[MIX_FEATS], y, XMIX_te[MIX_FEATS])
test_raw['lgb_mix_deep'] = lgb_predict(XMIX_tr[MIX_FEATS], y, XMIX_te[MIX_FEATS],
                                       n_estimators=1500, learning_rate=0.02,
                                       min_child_samples=20)
test_raw['cb_mix'] = cb_predict(XMIX_tr[MIX_FEATS], y, XMIX_te[MIX_FEATS], cat_names=CATS)
print('микс-тройка готова |', elapsed())

микс-тройка готова | 1225s


Собираем финальный скор: взвешенное среднее процентилей, NaN → 0.5, обрезка в [0, 1].

In [59]:
score = blend(test_raw, WEIGHTS)
score = np.nan_to_num(score, nan=0.5)
score = np.clip(score, 0.0, 1.0)

for name in WEIGHTS:
    print(f'{name:14s} вес={WEIGHTS[name]}  корреляция с ансамблем: '
          f'{np.corrcoef(rank_pct(test_raw[name]), score)[0, 1]:.4f}')

default        вес=1.0  корреляция с ансамблем: 0.9847
deep           вес=1.0  корреляция с ансамблем: 0.9849
leaves15       вес=1.0  корреляция с ансамблем: 0.9683
small          вес=1.0  корреляция с ансамблем: 0.9796
lgb_mix        вес=0.5  корреляция с ансамблем: 0.9294
lgb_mix_deep   вес=0.5  корреляция с ансамблем: 0.9292
cb_mix         вес=0.5  корреляция с ансамблем: 0.9227


### 6.1 Проверка файла ответа

Формат `submission.csv` жёстко проверяется хостером, поэтому прогоняем полный набор
assert-ов до записи: колонки как в `sample_submission`, строк ровно столько, сколько
в test, нет дублей и пропусков, состав cookie_id совпадает с test, score ∈ [0, 1].

In [61]:
submission = pd.DataFrame({'cookie_id': test.cookie_id.values, 'score': score})

sample = pd.read_csv('sample_submission.csv')
assert list(submission.columns) == list(sample.columns), 'колонки отличаются от образца'
assert len(submission) == len(test), 'число строк не совпадает с test'
assert submission.cookie_id.is_unique, 'дубли cookie_id'
assert set(submission.cookie_id) == set(test.cookie_id), 'состав cookie_id отличается от test'
assert submission.score.between(0, 1).all(), 'score вне [0, 1]'
assert submission.notna().all().all(), 'есть пропуски'

submission.to_csv(OUT, index=False)
print('сохранено', OUT, submission.shape)
print('разных значений score:', submission.score.nunique())
print(submission.score.describe().round(4))
print(submission.head())

сохранено submission.csv (4909, 2)
разных значений score: 4833
count    4909.0000
mean        0.5001
std         0.2786
min         0.0011
25%         0.2596
50%         0.4891
75%         0.7316
max         0.9994
Name: score, dtype: float64
             cookie_id     score
0  ck_315fb710a0e371e7  0.236245
1  ck_a76ee3b3e3e522fd  0.871516
2  ck_94c9a4d382689e82  0.758440
3  ck_8eaf9509ad9462a0  0.029389
4  ck_9a88a5a989cb5bc6  0.464916


In [62]:
# короткая диагностика: скор действительно выше для «ботоподобных» профилей
diagn = X_test.copy()
diagn['score'] = score
print('корреляция скоров с ключевыми признаками:')
corr = diagn[FEATS].corrwith(diagn['score']).abs().sort_values(ascending=False)
print(corr.head(12).round(3))
print('\nсредние значения у 10% кук с highest score:')
top = diagn.score >= diagn.score.quantile(0.9)
print(diagn.loc[top, ['n_events', 'page_max', 'cookie_age_days', 'loc_entropy',
                      'gap_med', 'night_share', 'ua_headless']].mean().round(2))
print('у остальных 90%:')
print(diagn.loc[~top, ['n_events', 'page_max', 'cookie_age_days', 'loc_entropy',
                       'gap_med', 'night_share', 'ua_headless']].mean().round(2))
print('\nготово за', elapsed())

корреляция скоров с ключевыми признаками:
px_std               0.463
px_range             0.451
py_std               0.384
px_mean              0.379
py_range             0.379
gap_med_pct          0.349
pointer_rate         0.340
events_per_minute    0.340
pointer_nuniq        0.338
py_mean              0.334
ptr_on_item          0.325
ptr_on_search        0.323
dtype: float64

средние значения у 10% кук с highest score:
n_events           31.66
page_max            9.96
cookie_age_days    43.23
loc_entropy         1.91
gap_med            80.68
night_share         0.22
ua_headless         0.07
dtype: float64
у остальных 90%:
n_events            16.78
page_max             3.97
cookie_age_days     74.80
loc_entropy          1.22
gap_med            208.17
night_share          0.13
ua_headless          0.02
dtype: float64

готово за 1225s


## 7. Что показали эксперименты

**Сработало и осталось в решении**

* **Walk-forward по дням** вместо случайного сплита: один и тот же сплит на 3 дня даёт
  разброс ±0.03 по метрике, 7 дней с OOF-скорами — заметно стабильнее.
* **Четыре слоя числовых признаков**: интенсивность/ритм потока, скриптовое поведение,
  доменные (популярность объектов внутри дня), среда (микс платформ и UA) плюс
  перцентили внутри дня. Отказ от признака `window_day` (календарный номер дня окна)
  убрал утечку «даты» и улучшил валидацию.
* **Слой категориальных признаков + микс-модели**: 15 категорий (наборы и доминанты
  платформ/событий, топ-товар/запрос/категория/локация, семейство UA и т.д.) в
  LightGBM (`category`) и CatBoost (`cat_features`). Поодиночке микс-модели слабее
  числовых (0.68–0.72 против 0.71–0.72), но коррелируют с ними лишь на 0.81 и в
  блендинге дают +0.02…+0.03: ens4 0.720 → финальный 7-модельный 0.745.
* **Взвешенное усреднение процентилей** (числа 1.0, микс 0.5): равные веса дают меньше,
  перевес микса тоже — устойчивое плато на 0.45–0.55.
* Baseline quickstart (два агрегата) даёт ~0.10 — признаки, а не модель, здесь главное.

**Проверено и отброшено**

* **CatBoost на одних категориях** (классический «блендинг по типам фич»):
  P@R0.7 = 0.13 — категории без числового контекста почти бесполезны, а блендинг
  CB(кат) + LGB(числ) = 0.36. Заработала только связка «категории + числа» в каждой
  модели.
* `lambdarank` (ранжирование с упором в верх списка) — заметно хуже бинарной цели;
* жёсткий отбор признаков по важности (top-30/top-60) — сильно хуже, чем все 240;
* CatBoost только на числовых (0.718) — включён в ансамбль только в «микс»-виде;
* «фабричные» подписи кук (похожие `n_events`/пауза/UA) — боты как раз **не** скапливаются
  в крупных группах, поэтому групповые мета-признаки не дали выигрыша;
* события вне окна (включая показы капчи) не использовали: по условию признаки берутся
  из суточного окна.

**Воспроизводимость**

* `random_state = 42` зафиксирован во всех моделях;
* версии окружения — в `requirements.txt` (Python 3.13);
* внешние API и LLM не использовались, только локальные open-source библиотеки
  `lightgbm`, `catboost`, `scikit-learn`, `pandas`, `numpy`;
* полный прогон ноутбука занимает ~20 минут и создаёт `submission.csv`.